<center> <h1>GenAI with Amazon Bedrock : Cohere Reranker</h1>

### 0. Preparation

#### 0.1 Bedrock Credentials

In [4]:
import yaml

with open('../secrets.yml', 'r') as file:
    credentials = yaml.safe_load(file)

#### 0.2 Sample Data

In [5]:
documents = [
    "According to the Carbon Majors database, the main contributors to GHG emissions and their role in global warming are fossil fuel companies. These companies, both state-owned and private, have produced almost a trillion tons of GHG emissions in 150 years. The database shows that 100 existing fossil fuel companies, along with eight that no longer exist, are responsible for 71% of all GHG emissions since 1988. In the Americas, the private companies that have contributed the most emissions are ExxonMobil, Chevron, and Peabody, all from the United States. Among state-owned companies in the Americas, the largest emitter is Mexican company Pemex, followed by Venezuelan company Petróleos de Venezuela, S.A. It is important to note that while people with fewer resources, particularly from countries in the global South, do not significantly contribute to climate change, they are the ones most affected by its impacts. Approximately half of the global population lives in areas that are 'very vulnerable' to climate change, and it is people with limited development opportunities who face the greatest risks. This unequal impact disproportionately affects the human rights of those with fewer resources and greater vulnerability in the context of climate change in the global South. Additionally, between 2010 and 2020, human mortality due to climate disasters was 15 times higher in vulnerable regions and populations.", 
    "The largest private companies in the Americas that are the largest GHG emitters according to the Carbon Majors database are ExxonMobil, Chevron, and Peabody.", 
    "Amnesty International urged its supporters to send appeals for the defenders' freedom to Nigerian authorities and later to send letters of outrage.", 
    "The recommendations made by Amnesty International to the Special Rapporteur on Human Rights Defenders include embedding a focus on child and young HRDs in future work, raising awareness about the differences and challenges they face, incorporating age disaggregated data in reports, and creating safe spaces for engagement.", 
    "The target audience of the two books created by Amnesty International on child rights are children and young people.",
    "The right that guarantees access to comprehensive information about past human rights violations, including the identities of the perpetrators and the fate of the victims, as well as the circumstances surrounding the violations, is the right to know the truth.", 
    "The victims of gross human rights violations and their families, as well as members of society generally, have the right to be fully informed about human rights violations, including the identities of the perpetrators and the fate of the victims.", 
    "Individuals can be found guilty under Article 207.3 of the Russian Criminal Code if their statements are contrary to the official position of the Russian authorities.", 
    "The prosecution considers statements contrary to the official position as 'false' under Article 207.3 when they are in opposition to the official position of the Russian authorities.", 
    "The factors that have contributed to the decline of independent civil society organizations in Nicaragua include arrests and harassment of human rights defenders, restrictive NGO laws, violent repression of protests, closure of civil society organizations and community centers, expropriation of belongings and premises, criminalization of social organizing and mobilization, restrictions on freedom of expression, implementation of repressive laws, constant threats of arrest and detention, restrictions on social media, and the imposition of silence through violence and repression.", 
    "The conditions that designate wetlands as Ramsar sites are when they fulfill the criteria for identifying wetlands of international importance, as established under the Convention on Wetlands.", 
    "COP15 was held in Montreal, Canada in 2022.", 
    "The States failed to explicitly recognize Indigenous Peoples' lands and territories as a distinct category of protected area at COP15.", 
    "The consequences of criminalizing abortion for marginalized individuals include increased stigma, lack of information, and disinformation. This can have severe and irreversible effects on these individuals. Girls and young women may be forced to carry pregnancies resulting from sexual violence due to a lack of knowledge about their rights. Marginalized individuals, such as those living in poverty, historically discriminated against, Indigenous and Afro-descendent women, migrants, and refugees, are disproportionately affected by abortion criminalization. The criminalization of abortion is a major factor contributing to the high number of unsafe abortions, which leads to increased maternal mortality and morbidity. Access to health services is undermined, resulting in preventable maternal deaths and complications. Marginalized individuals are forced to resort to unsafe clandestine abortion methods, putting their lives and health at risk. In Nigeria, restrictive abortion laws make it difficult to access safe abortion care.", 
    "Social media companies should have the responsibility to invest in human oversight of their content moderation systems to ensure equal access to accurate sexual and reproductive health information. They should also engage in human rights due diligence to address risks and abuses related to their business model.", 
    "Social media companies play a role in protecting users' rights online, regardless of their language and political views, by investing in human oversight of content moderation systems, engaging in human rights due diligence, and educating users about security and privacy features.", 
    "Amnesty International documented labor abuses in Qatar, including workers being permitted only one day off each month, threats of salary cuts for taking more rest time, failure to provide pay slips, and overcrowded and dirty living conditions. These labor abuses relate to the kafala system because the kafala system imposes tight restrictions on migrant workers' freedom of movement and ability to change jobs without their employer's permission. This system allows abusive employers to control their workforce by canceling residence permits, falsely reporting employees as absconding, and using non-compete clauses to prevent workers from changing jobs.", 
    "The government of Qatar started repealing restrictions on migrant workers between 2018 and 2020."
]

### 1. Direct Call Through Bedrock Client

##### Method #1 : Cohere Bedrock Client

In [20]:
%pip install cohere

   ---------------------------------------- 0.0/252.5 kB ? eta -:--:--
   ---- ----------------------------------- 30.7/252.5 kB 1.3 MB/s eta 0:00:01
   ---------------------------------------- 252.5/252.5 kB 3.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/499.8 kB ? eta -:--:--
   -------------------------- ------------- 327.7/499.8 kB 6.8 MB/s eta 0:00:01
   ---------------------------------------- 499.8/499.8 kB 5.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.4 MB 5.9 MB/s eta 0:00:01
   -------- ------------------------------- 0.5/2.4 MB 6.3 MB/s eta 0:00:01
   ------------- -------------------------- 0.8/2.4 MB 6.1 MB/s eta 0:00:01
   ------------------ --------------------- 1.1/2.4 MB 6.2 MB/s eta 0:00:01
   ---------------------- ----------------- 1.3/2.4 MB 6.0 MB/s eta 0:00:01
   ------------------------ --------------- 1.4/2.4 MB 5.7 MB/s eta 0:00:01
   -----------


[notice] A new release of pip is available: 24.0 -> 25.0
[notice] To update, run: C:\Users\Admin\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [6]:
import cohere

In [7]:
cohere_bedrock_client = cohere.BedrockClientV2(
    aws_region="eu-central-1",
    aws_access_key=credentials["bedrock"]["access_key"],
    aws_secret_key=credentials["bedrock"]["secret_key"]
)

In [24]:
cohere_bedrock_response = cohere_bedrock_client.rerank(
    query="Where was COP15 held in 2022?",
    documents=documents,
    model="cohere.rerank-v3-5:0",
    top_n=5
)

In [36]:
cohere_bedrock_response.results

[V2RerankResponseResultsItem(document=None, index=11, relevance_score=0.95386744),
 V2RerankResponseResultsItem(document=None, index=12, relevance_score=0.08368814),
 V2RerankResponseResultsItem(document=None, index=0, relevance_score=0.07480398),
 V2RerankResponseResultsItem(document=None, index=16, relevance_score=0.02093424),
 V2RerankResponseResultsItem(document=None, index=17, relevance_score=0.018322192)]

In [35]:
[documents[res.index] for res in cohere_bedrock_response.results]

['COP15 was held in Montreal, Canada in 2022.',
 "The States failed to explicitly recognize Indigenous Peoples' lands and territories as a distinct category of protected area at COP15.",
 "According to the Carbon Majors database, the main contributors to GHG emissions and their role in global warming are fossil fuel companies. These companies, both state-owned and private, have produced almost a trillion tons of GHG emissions in 150 years. The database shows that 100 existing fossil fuel companies, along with eight that no longer exist, are responsible for 71% of all GHG emissions since 1988. In the Americas, the private companies that have contributed the most emissions are ExxonMobil, Chevron, and Peabody, all from the United States. Among state-owned companies in the Americas, the largest emitter is Mexican company Pemex, followed by Venezuelan company Petróleos de Venezuela, S.A. It is important to note that while people with fewer resources, particularly from countries in the glob

##### Method #2 : Pure Bedrock Client

In [9]:
import boto3
import json

In [10]:
pure_bedrock_client = boto3.client(
    "bedrock-runtime",
    region_name="eu-central-1",
    aws_access_key_id=credentials["bedrock"]["access_key"],
    aws_secret_access_key=credentials["bedrock"]["secret_key"]
)

In [52]:
pure_bedrock_response = pure_bedrock_client.invoke_model(
    modelId="cohere.rerank-v3-5:0",
    body=json.dumps(
        {
            "query": "Where was COP15 held in 2022?",
            "documents": documents,
            "top_n": 5,
            "api_version": 2
        }
    )
)

In [53]:
decoded_response = json.loads(pure_bedrock_response["body"].read().decode("utf-8"))

In [54]:
decoded_response["results"]

[{'index': 11, 'relevance_score': 0.95386744},
 {'index': 12, 'relevance_score': 0.08368814},
 {'index': 0, 'relevance_score': 0.07480398},
 {'index': 16, 'relevance_score': 0.02093424},
 {'index': 17, 'relevance_score': 0.018322192}]

In [56]:
[documents[res["index"]] for res in decoded_response["results"]]

['COP15 was held in Montreal, Canada in 2022.',
 "The States failed to explicitly recognize Indigenous Peoples' lands and territories as a distinct category of protected area at COP15.",
 "According to the Carbon Majors database, the main contributors to GHG emissions and their role in global warming are fossil fuel companies. These companies, both state-owned and private, have produced almost a trillion tons of GHG emissions in 150 years. The database shows that 100 existing fossil fuel companies, along with eight that no longer exist, are responsible for 71% of all GHG emissions since 1988. In the Americas, the private companies that have contributed the most emissions are ExxonMobil, Chevron, and Peabody, all from the United States. Among state-owned companies in the Americas, the largest emitter is Mexican company Pemex, followed by Venezuelan company Petróleos de Venezuela, S.A. It is important to note that while people with fewer resources, particularly from countries in the glob

### 2. Langchain Integration

#### 2.1 Vanilla Retriever

##### 2.1.1 Local Embedding on LM Studio

In [32]:
%pip install langchain-openai langchain-qdrant

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.0
[notice] To update, run: C:\Users\Admin\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [12]:
from langchain_openai import OpenAIEmbeddings

embedding = OpenAIEmbeddings(
    openai_api_base="http://localhost:1234/v1", 
    api_key="type-anything-here",
    model="text-embedding-bge-large-en-v1.5",
    check_embedding_ctx_length=False
)

##### 2.1.2 Qdrant Vectorstore

In [13]:
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore, RetrievalMode
from langchain_core.documents import Document

qdrant_clt = QdrantClient(":memory:")

qdrant_clt.create_collection(
    collection_name="embedding_tutorial",
    vectors_config=VectorParams(size=1024, distance=Distance.COSINE),
)

qdrant = QdrantVectorStore(
    embedding=embedding,
    client=qdrant_clt,
    collection_name="embedding_tutorial",
    retrieval_mode=RetrievalMode.DENSE,
)

In [14]:
qdrant.add_documents(
    [
        Document(page_content=doc, metadata={})
        for doc in documents
    ]
)

['7234da4cffde42e6808ecb2e30f77f96',
 '2fe678947d424f5a9bee6a6a7035a212',
 '1682542e0fa94a63adf286fdf3fe83c2',
 '14bf4958804e449ebdec617fc47ec526',
 '40ab01a11e934810ae41ba4634298474',
 'fad0521ae6284012ab0faf5f895904e3',
 'ace418617839407ea2479c8417b03162',
 'dc0e2f72dd9b4e788e07eacfee88785f',
 '0804e09e3aee4862bf96ee9c03002226',
 '3248eac1543c45f7a38466e41fe51d79',
 '0c1f00eeeaad4c61a25d3d2b57f10621',
 'dac9a34e34d54dc4b27dab76ad1e07dc',
 '6e3dca6d742f46919adbc6360141f2e2',
 '9bdb8f2ff9bd460087d8c5d83b4f266a',
 'd74c0d954fb6430c8a356fd49ca577d4',
 '0ebe6215c43849dca502676bf85da9bf',
 '790ddf3f2e7140cc83e6f0b749898e2f',
 '2dff796afdbe40ce908dd2bf79bb9993']

##### 2.1.3 Retriever

In [15]:
retriever = qdrant.as_retriever(search_kwargs={"k": 5})

In [16]:
retriever.invoke("Where was COP15 held in 2022?")

[Document(metadata={'_id': 'dac9a34e34d54dc4b27dab76ad1e07dc', '_collection_name': 'embedding_tutorial'}, page_content='COP15 was held in Montreal, Canada in 2022.'),
 Document(metadata={'_id': '6e3dca6d742f46919adbc6360141f2e2', '_collection_name': 'embedding_tutorial'}, page_content="The States failed to explicitly recognize Indigenous Peoples' lands and territories as a distinct category of protected area at COP15."),
 Document(metadata={'_id': '7234da4cffde42e6808ecb2e30f77f96', '_collection_name': 'embedding_tutorial'}, page_content="According to the Carbon Majors database, the main contributors to GHG emissions and their role in global warming are fossil fuel companies. These companies, both state-owned and private, have produced almost a trillion tons of GHG emissions in 150 years. The database shows that 100 existing fossil fuel companies, along with eight that no longer exist, are responsible for 71% of all GHG emissions since 1988. In the Americas, the private companies that 

#### 2.2 Enhanced Retriever

##### 2.2.1 Rerank Runnable

In [61]:
from langchain_core.runnables import RunnableLambda

def rerank_cohere(input):
    query = input["query"]
    documents = input["documents"]
    response = cohere_bedrock_client.rerank(
        query=query, 
        documents=[doc.page_content for doc in documents], 
        model="cohere.rerank-v3-5:0", 
        top_n=3
    )
    return [documents[res.index] for res in response.results]

reranker = RunnableLambda(rerank_cohere)

##### 2.2.2 Full Chain

In [62]:
from langchain_core.runnables import RunnablePassthrough

enhanced_retriever = {"query": RunnablePassthrough(), "documents": retriever} | reranker

In [63]:
enhanced_retriever.invoke("Where was COP15 held in 2022?")

[Document(metadata={'_id': 'dac9a34e34d54dc4b27dab76ad1e07dc', '_collection_name': 'embedding_tutorial'}, page_content='COP15 was held in Montreal, Canada in 2022.'),
 Document(metadata={'_id': '6e3dca6d742f46919adbc6360141f2e2', '_collection_name': 'embedding_tutorial'}, page_content="The States failed to explicitly recognize Indigenous Peoples' lands and territories as a distinct category of protected area at COP15."),
 Document(metadata={'_id': '7234da4cffde42e6808ecb2e30f77f96', '_collection_name': 'embedding_tutorial'}, page_content="According to the Carbon Majors database, the main contributors to GHG emissions and their role in global warming are fossil fuel companies. These companies, both state-owned and private, have produced almost a trillion tons of GHG emissions in 150 years. The database shows that 100 existing fossil fuel companies, along with eight that no longer exist, are responsible for 71% of all GHG emissions since 1988. In the Americas, the private companies that 